# 01 — Exploratory Data Analysis

MDQ 2026 — поиск скрытых предпринимателей среди физлиц.

Цель ноутбука: понять структуру данных, сравнить business vs consumer, проверить качество, подтвердить/опровергнуть гипотезы из `docs/FEATURES.md`.

In [1]:
# Импорты и настройка путей
import sys
from pathlib import Path

# Добавляем корень репо в sys.path чтобы импортировать src.config
ROOT = Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import polars as pl
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from src.config import (
    BUSINESS_CARDS_PATH, CONSUMER_CARDS_PATH, MERCHANTS_PATH,
    BUSINESS_HOURS_START, BUSINESS_HOURS_END,
    B2B_MCC_CODES, CONSUMER_MCC_CODES, RANDOM_STATE,
)

np.random.seed(RANDOM_STATE)
pl.Config.set_tbl_rows(20)
pl.Config.set_tbl_cols(20)
print('Polars version:', pl.__version__)

Polars version: 1.41.0


## 1. Загрузка данных

In [2]:
business = pl.read_parquet(BUSINESS_CARDS_PATH)
consumer = pl.read_parquet(CONSUMER_CARDS_PATH)
merchants = pl.read_parquet(MERCHANTS_PATH)

print(f'business : {business.shape}')
print(f'consumer : {consumer.shape}')
print(f'merchants: {merchants.shape}')

business : (2997593, 12)
consumer : (9832487, 12)
merchants: (2165, 5)


In [3]:
# Нормализуем имя поля is_recurring (в схеме CASE.md оно с большой буквы)
for name in ['business', 'consumer']:
    df = business if name == 'business' else consumer
    if 'Is_recurring' in df.columns and 'is_recurring' not in df.columns:
        df = df.rename({'Is_recurring': 'is_recurring'})
        if name == 'business':
            business = df
        else:
            consumer = df
print('business cols:', business.columns)
print('consumer cols:', consumer.columns)
print('merchants cols:', merchants.columns)

business cols: ['transaction_date', 'transaction_timestamp', 'transaction_amount_kzt', 'mcc', 'merchant_id', 'channel', 'bank_name', 'country', 'card_number', 'card_tier', 'tokenized', 'is_recurring']
consumer cols: ['transaction_date', 'transaction_timestamp', 'transaction_amount_kzt', 'mcc', 'merchant_id', 'channel', 'bank_name', 'country', 'card_number', 'card_tier', 'tokenized', 'is_recurring']
merchants cols: ['merchant_id', 'merchant_name', 'mcc', 'merchant_country', 'recurring_capable']


## 2. Базовый обзор каждого датасета

In [4]:
def overview(df: pl.DataFrame, name: str) -> None:
    """Печатает базовую статистику по датасету."""
    print(f'\n=== {name} ===')
    print(f'shape : {df.shape}')
    print('\n-- dtypes --')
    for col, dt in zip(df.columns, df.dtypes):
        print(f'  {col:30s} {dt}')
    print('\n-- null counts --')
    nulls = df.null_count().to_dicts()[0]
    any_null = False
    for col, n in nulls.items():
        if n > 0:
            print(f'  {col:30s} {n:>10,}')
            any_null = True
    if not any_null:
        print('  (no nulls)')

overview(business, 'business_cards')
overview(consumer, 'consumer_cards')
overview(merchants, 'merchants_reference')


=== business_cards ===
shape : (2997593, 12)

-- dtypes --
  transaction_date               Date
  transaction_timestamp          Datetime(time_unit='ms', time_zone=None)
  transaction_amount_kzt         Int64
  mcc                            String
  merchant_id                    String
  channel                        String
  bank_name                      String
  country                        String
  card_number                    String
  card_tier                      String
  tokenized                      Boolean
  is_recurring                   Boolean

-- null counts --
  (no nulls)

=== consumer_cards ===
shape : (9832487, 12)

-- dtypes --
  transaction_date               Date
  transaction_timestamp          Datetime(time_unit='ms', time_zone=None)
  transaction_amount_kzt         Int64
  mcc                            String
  merchant_id                    String
  channel                        String
  bank_name                      String
  country               

In [5]:
business.head(10)

transaction_date,transaction_timestamp,transaction_amount_kzt,mcc,merchant_id,channel,bank_name,country,card_number,card_tier,tokenized,is_recurring
date,datetime[ms],i64,str,str,str,str,str,str,str,bool,bool
2025-10-01,2025-10-01 00:00:00,180976,"""7372""","""MER_000007""","""online""","""Kaspi""","""US""","""5228592291438845""","""Business""",false,true
2025-10-01,2025-10-01 00:00:00,153206,"""7372""","""MER_000006""","""online""","""Home Credit Bank""","""US""","""5201495142193372""","""Business""",false,true
2025-10-01,2025-10-01 00:00:00,197106,"""7372""","""MER_000007""","""online""","""Home Credit Bank""","""US""","""5201492177677288""","""Business""",false,true
2025-10-01,2025-10-01 00:01:00,189598,"""7372""","""MER_000008""","""online""","""Kaspi""","""US""","""5176513443697635""","""Business""",false,true
2025-10-01,2025-10-01 00:03:00,700571,"""7311""","""MER_000003""","""online""","""Halyk""","""Russia""","""5100611967455520""","""Business""",false,true
2025-10-01,2025-10-01 00:04:00,228899,"""7372""","""MER_000009""","""online""","""Eurasian Bank""","""US""","""5441026246199534""","""Business""",false,true
2025-10-01,2025-10-01 00:05:00,206896,"""7372""","""MER_000009""","""online""","""Eurasian Bank""","""US""","""5441029193739260""","""Business""",false,true
2025-10-01,2025-10-01 00:06:00,391854,"""7311""","""MER_000002""","""online""","""Halyk""","""Singapore""","""5100610143748493""","""Business""",false,true
2025-10-01,2025-10-01 00:07:00,315274,"""4816""","""MER_000883""","""online""","""Eurasian Bank""","""Kazakhstan""","""5441024234542971""","""Business""",false,true


In [6]:
consumer.head(10)

transaction_date,transaction_timestamp,transaction_amount_kzt,mcc,merchant_id,channel,bank_name,country,card_number,card_tier,tokenized,is_recurring
date,datetime[ms],i64,str,str,str,str,str,str,str,bool,bool
2025-10-01,2025-10-01 00:04:00,4788,"""4814""","""MER_000064""","""online""","""Alatau City Bank""","""Kazakhstan""","""5263907968824596""","""Standard""",false,true
2025-10-01,2025-10-01 00:10:00,5240,"""4814""","""MER_000063""","""online""","""Bank RBK""","""Kazakhstan""","""5119023663984986""","""Standard""",false,true
2025-10-01,2025-10-01 00:12:00,4576,"""4814""","""MER_000066""","""online""","""Kaspi""","""Kazakhstan""","""5228590878155154""","""Standard""",false,true
2025-10-01,2025-10-01 00:37:00,6078,"""4814""","""MER_000063""","""online""","""Home Credit Bank""","""Kazakhstan""","""5338472125333693""","""Standard""",false,true
2025-10-01,2025-10-01 00:37:00,6042,"""4814""","""MER_000065""","""online""","""Kaspi""","""Kazakhstan""","""5531514712394557""","""Affluent""",false,true
2025-10-01,2025-10-01 00:46:17,137960,"""7995""","""MER_000849""","""online""","""Kaspi""","""Kazakhstan""","""5176512180841174""","""Standard""",true,false
2025-10-01,2025-10-01 00:53:07,122057,"""5099""","""MER_001015""","""online""","""Eurasian Bank""","""Kazakhstan""","""5441020083063408""","""Affluent""",true,false
2025-10-01,2025-10-01 01:15:53,226475,"""8211""","""MER_000606""","""online""","""Bank RBK""","""Kazakhstan""","""5119023509211073""","""Standard""",false,false
2025-10-01,2025-10-01 01:36:00,4274,"""4814""","""MER_000066""","""online""","""Kaspi""","""Kazakhstan""","""5228598880636921""","""Standard""",false,true


In [7]:
merchants.head(10)

merchant_id,merchant_name,mcc,merchant_country,recurring_capable
str,str,str,str,bool
"""MER_000000""","""Google Ads""","""7311""","""Ireland""",true
"""MER_000001""","""Meta Ads""","""7311""","""Ireland""",true
"""MER_000002""","""TikTok Ads""","""7311""","""Singapore""",true
"""MER_000003""","""Yandex Direct""","""7311""","""Russia""",true
"""MER_000004""","""LinkedIn Ads""","""7311""","""Ireland""",true
"""MER_000005""","""Instagram Promote""","""7311""","""Ireland""",true
"""MER_000006""","""Amazon Web Services""","""7372""","""US""",true
"""MER_000007""","""Microsoft Azure""","""7372""","""US""",true
"""MER_000008""","""Google Cloud""","""7372""","""US""",true


In [8]:
business.describe()

statistic,transaction_date,transaction_timestamp,transaction_amount_kzt,mcc,merchant_id,channel,bank_name,country,card_number,card_tier,tokenized,is_recurring
str,str,str,f64,str,str,str,str,str,str,str,f64,f64
"""count""","""2997593""","""2997593""",2.997593e6,"""2997593""","""2997593""","""2997593""","""2997593""","""2997593""","""2997593""","""2997593""",2.997593e6,2.997593e6
"""null_count""","""0""","""0""",0.0,"""0""","""0""","""0""","""0""","""0""","""0""","""0""",0.0,0.0
"""mean""","""2026-01-01 10:57:48.976075""","""2026-01-01 23:22:32.938000""",156535.274681,null,null,null,null,null,null,null,0.600008,0.133447
"""std""",null,null,252868.24236,null,null,null,null,null,null,null,null,null
"""min""","""2025-10-01""","""2025-10-01 00:00:00""",67.0,"""2741""","""MER_000000""","""POS""","""Alatau City Bank""","""Australia""","""5100610003025081""","""Business""",0.0,0.0
"""25%""","""2025-11-17""","""2025-11-17 15:42:00""",22835.0,null,null,null,null,null,null,null,null,null
"""50%""","""2026-01-01""","""2026-01-01 13:32:36""",77224.0,null,null,null,null,null,null,null,null,null
"""75%""","""2026-02-17""","""2026-02-17 16:16:51""",196081.0,null,null,null,null,null,null,null,null,null
"""max""","""2026-03-31""","""2026-03-31 23:59:53""",4.0799297e7,"""8931""","""MER_002153""","""online""","""Kaspi""","""Uzbekistan""","""5531519996149148""","""Business""",1.0,1.0


In [9]:
consumer.describe()

statistic,transaction_date,transaction_timestamp,transaction_amount_kzt,mcc,merchant_id,channel,bank_name,country,card_number,card_tier,tokenized,is_recurring
str,str,str,f64,str,str,str,str,str,str,str,f64,f64
"""count""","""9832487""","""9832487""",9.832487e6,"""9832487""","""9832487""","""9832487""","""9832487""","""9832487""","""9832487""","""9832487""",9.832487e6,9.832487e6
"""null_count""","""0""","""0""",0.0,"""0""","""0""","""0""","""0""","""0""","""0""","""0""",0.0,0.0
"""mean""","""2025-12-30 02:12:35.726033""","""2025-12-30 17:42:42.777000""",54045.424791,null,null,null,null,null,null,null,0.386253,0.02718
"""std""",null,null,169654.627229,null,null,null,null,null,null,null,null,null
"""min""","""2025-10-01""","""2025-10-01 00:00:00""",15.0,"""0780""","""MER_000000""","""POS""","""Alatau City Bank""","""Australia""","""5100610005930965""","""Affluent""",0.0,0.0
"""25%""","""2025-11-16""","""2025-11-16 23:02:47""",4172.0,null,null,null,null,null,null,null,null,null
"""50%""","""2025-12-28""","""2025-12-28 09:27:40""",11892.0,null,null,null,null,null,null,null,null,null
"""75%""","""2026-02-13""","""2026-02-13 18:44:31""",39665.0,null,null,null,null,null,null,null,null,null
"""max""","""2026-03-31""","""2026-03-31 23:59:49""",3.1971032e7,"""9700""","""MER_002164""","""online""","""Kaspi""","""Uzbekistan""","""5531519994943930""","""Standard""",1.0,1.0


In [10]:
merchants.describe()

statistic,merchant_id,merchant_name,mcc,merchant_country,recurring_capable
str,str,str,str,str,f64
"""count""","""2165""","""2165""","""2165""","""2165""",2165.0
"""null_count""","""0""","""0""","""0""","""0""",0.0
"""mean""",null,null,null,null,0.012471
"""std""",null,null,null,null,null
"""min""","""MER_000000""","""Accounting/Auditin_3186""","""0780""","""Australia""",0.0
"""25%""",null,null,null,null,null
"""50%""",null,null,null,null,null
"""75%""",null,null,null,null,null
"""max""","""MER_002164""","""inDrive""","""9700""","""US""",1.0


## 3. Сравнение business vs consumer

Все графики строим в Plotly. Для распределений берём sample транзакций — на полном объёме гистограммы рисуются долго.

### 3.1 Распределение transaction_amount_kzt (log scale)

In [12]:
# Сэмплируем по 300K транзакций из каждого класса — гистограмма не теряет форму
SAMPLE_N = 300_000
b_sample = business.sample(n=min(SAMPLE_N, business.height), seed=RANDOM_STATE)
c_sample = consumer.sample(n=min(SAMPLE_N, consumer.height), seed=RANDOM_STATE)

amt_df = pl.concat([
    b_sample.select(pl.col('transaction_amount_kzt')).with_columns(pl.lit('business').alias('card_type')),
    c_sample.select(pl.col('transaction_amount_kzt')).with_columns(pl.lit('consumer').alias('card_type')),
]).filter(pl.col('transaction_amount_kzt') > 0).to_pandas()

fig = px.histogram(
    amt_df, x='transaction_amount_kzt', color='card_type',
    nbins=80, log_x=True, log_y=True, barmode='overlay', opacity=0.55,
    title='Распределение transaction_amount_kzt (log-log, sample 300K на класс)',
)
fig.update_layout(xaxis_title='Сумма транзакции, ₸ (log)', yaxis_title='Частота (log)')
fig.show()

### 3.2 Агрегаты на уровне карты

In [13]:
# Считаем по каждой карте: n_tx, n_unique_merchants, n_unique_mcc, online_share, tokenized_share, recurring_share
def card_aggregates(df: pl.DataFrame, label: str) -> pl.DataFrame:
    return (
        df.group_by('card_number').agg([
            pl.len().alias('n_tx'),
            pl.col('merchant_id').n_unique().alias('n_unique_merchants'),
            pl.col('mcc').n_unique().alias('n_unique_mcc'),
            (pl.col('channel') == 'online').mean().alias('online_share'),
            pl.col('tokenized').cast(pl.Float64).mean().alias('tokenized_share'),
            pl.col('is_recurring').cast(pl.Float64).mean().alias('recurring_share'),
            pl.col('transaction_amount_kzt').sum().alias('total_spend'),
        ])
        .with_columns(pl.lit(label).alias('card_type'))
    )

b_cards = card_aggregates(business, 'business')
c_cards = card_aggregates(consumer, 'consumer')
cards = pl.concat([b_cards, c_cards])
print('Карт всего:', cards.shape)
cards.group_by('card_type').agg([
    pl.col('n_tx').median().alias('median_n_tx'),
    pl.col('n_unique_merchants').median().alias('median_n_merch'),
    pl.col('n_unique_mcc').median().alias('median_n_mcc'),
    pl.col('online_share').mean().alias('mean_online_share'),
    pl.col('tokenized_share').mean().alias('mean_tokenized'),
    pl.col('recurring_share').mean().alias('mean_recurring'),
    pl.col('total_spend').median().alias('median_total_spend'),
])

Карт всего: (105000, 9)


card_type,median_n_tx,median_n_merch,median_n_mcc,mean_online_share,mean_tokenized,mean_recurring,median_total_spend
str,f64,f64,f64,f64,f64,f64,f64
"""consumer""",120.0,37.0,32.0,0.468678,0.38478,0.031421,2.976294e6
"""business""",119.0,16.0,15.0,0.848435,0.584569,0.1547,1.7715e7


In [14]:
# Боксплоты сравнения на уровне карты (log y где имеет смысл)
cards_pd = cards.to_pandas()
metrics = [
    ('n_tx', 'Транзакций на карту', True),
    ('n_unique_merchants', 'Уникальных мерчантов', True),
    ('n_unique_mcc', 'Уникальных MCC', False),
]
fig = make_subplots(rows=1, cols=3, subplot_titles=[m[1] for m in metrics])
for i, (col, _title, logy) in enumerate(metrics, start=1):
    for ct, color in [('business', '#1f77b4'), ('consumer', '#ff7f0e')]:
        fig.add_trace(
            go.Box(y=cards_pd.loc[cards_pd['card_type'] == ct, col],
                   name=ct, marker_color=color, showlegend=(i == 1)),
            row=1, col=i,
        )
    if logy:
        fig.update_yaxes(type='log', row=1, col=i)
fig.update_layout(height=420, title_text='Card-level метрики: business vs consumer')
fig.show()

In [15]:
# Доли (online / tokenized / recurring) — средние по картам
share_df = (
    cards.group_by('card_type').agg([
        pl.col('online_share').mean(),
        pl.col('tokenized_share').mean(),
        pl.col('recurring_share').mean(),
    ])
    .unpivot(index='card_type', variable_name='metric', value_name='share')
    .to_pandas()
)
fig = px.bar(share_df, x='metric', y='share', color='card_type', barmode='group',
             title='Средние доли: online / tokenized / recurring (по картам)')
fig.update_layout(yaxis_tickformat='.0%')
fig.show()

### 3.3 Топ-20 MCC у business vs consumer

In [16]:
def top_mcc(df: pl.DataFrame, k: int = 20) -> pl.DataFrame:
    return (
        df.group_by('mcc').agg(pl.len().alias('n_tx'))
          .sort('n_tx', descending=True)
          .head(k)
    )

top_b = top_mcc(business).with_columns(pl.lit('business').alias('card_type'))
top_c = top_mcc(consumer).with_columns(pl.lit('consumer').alias('card_type'))
top_mcc_df = pl.concat([top_b, top_c]).to_pandas()
top_mcc_df['mcc'] = top_mcc_df['mcc'].astype(str)

fig = px.bar(top_mcc_df, x='mcc', y='n_tx', color='card_type', barmode='group',
             title='Топ-20 MCC по количеству транзакций (business vs consumer)')
fig.update_layout(xaxis={'categoryorder': 'total descending'})
fig.show()

### 3.4 Временные паттерны — часы дня, дни недели, месяцы

In [17]:
def add_time_parts(df: pl.DataFrame) -> pl.DataFrame:
    return df.with_columns([
        pl.col('transaction_timestamp').dt.hour().alias('hour'),
        pl.col('transaction_timestamp').dt.weekday().alias('dow'),
        pl.col('transaction_timestamp').dt.truncate('1mo').alias('month'),
    ])

b_t = add_time_parts(business)
c_t = add_time_parts(consumer)

def share_by(df: pl.DataFrame, col: str, label: str) -> pl.DataFrame:
    total = df.height
    return (df.group_by(col).agg(pl.len().alias('n'))
              .with_columns([(pl.col('n') / total).alias('share'),
                             pl.lit(label).alias('card_type')])
              .sort(col))

hours = pl.concat([share_by(b_t, 'hour', 'business'),
                   share_by(c_t, 'hour', 'consumer')]).to_pandas()
fig = px.line(hours, x='hour', y='share', color='card_type', markers=True,
              title='Распределение транзакций по часам дня')
fig.add_vrect(x0=BUSINESS_HOURS_START, x1=BUSINESS_HOURS_END,
              fillcolor='lightgreen', opacity=0.2, line_width=0,
              annotation_text='Рабочие часы')
fig.update_layout(yaxis_tickformat='.1%')
fig.show()

In [18]:
dow = pl.concat([share_by(b_t, 'dow', 'business'),
                 share_by(c_t, 'dow', 'consumer')]).to_pandas()
dow_names = {1: 'Пн', 2: 'Вт', 3: 'Ср', 4: 'Чт', 5: 'Пт', 6: 'Сб', 7: 'Вс'}
dow['day'] = dow['dow'].map(dow_names)
fig = px.bar(dow, x='day', y='share', color='card_type', barmode='group',
             category_orders={'day': list(dow_names.values())},
             title='Распределение транзакций по дням недели')
fig.update_layout(yaxis_tickformat='.1%')
fig.show()

In [19]:
month = pl.concat([share_by(b_t, 'month', 'business'),
                   share_by(c_t, 'month', 'consumer')]).to_pandas()
fig = px.line(month, x='month', y='share', color='card_type', markers=True,
              title='Распределение транзакций по месяцам (10.2025 — 03.2026)')
fig.update_layout(yaxis_tickformat='.1%')
fig.show()

## 4. Анализ мерчантов

In [20]:
def top_merchants(df: pl.DataFrame, k: int = 20) -> pl.DataFrame:
    return (
        df.group_by('merchant_id').agg(pl.len().alias('n_tx'))
          .sort('n_tx', descending=True)
          .head(k)
          .join(merchants.select(['merchant_id', 'merchant_name', 'mcc']),
                on='merchant_id', how='left')
    )

top_b_m = top_merchants(business)
top_c_m = top_merchants(consumer)
print('Топ-20 мерчантов у business:')
print(top_b_m)
print('\nТоп-20 мерчантов у consumer:')
print(top_c_m)

overlap = set(top_b_m['merchant_id']) & set(top_c_m['merchant_id'])
print(f'\nПересечение топ-20: {len(overlap)} мерчантов')
if overlap:
    print(merchants.filter(pl.col('merchant_id').is_in(list(overlap))))

Топ-20 мерчантов у business:
shape: (20, 4)
┌─────────────┬───────┬───────────────────────┬──────┐
│ merchant_id ┆ n_tx  ┆ merchant_name         ┆ mcc  │
│ ---         ┆ ---   ┆ ---                   ┆ ---  │
│ str         ┆ u32   ┆ str                   ┆ str  │
╞═════════════╪═══════╪═══════════════════════╪══════╡
│ MER_000000  ┆ 63450 ┆ Google Ads            ┆ 7311 │
│ MER_000001  ┆ 46002 ┆ Meta Ads              ┆ 7311 │
│ MER_000003  ┆ 45372 ┆ Yandex Direct         ┆ 7311 │
│ MER_000005  ┆ 45234 ┆ Instagram Promote     ┆ 7311 │
│ MER_000002  ┆ 44904 ┆ TikTok Ads            ┆ 7311 │
│ MER_000004  ┆ 44266 ┆ LinkedIn Ads          ┆ 7311 │
│ MER_000010  ┆ 42791 ┆ HubSpot               ┆ 7372 │
│ MER_000006  ┆ 42108 ┆ Amazon Web Services   ┆ 7372 │
│ MER_000007  ┆ 40903 ┆ Microsoft Azure       ┆ 7372 │
│ MER_000021  ┆ 38849 ┆ Hetzner               ┆ 4816 │
│ MER_000020  ┆ 38646 ┆ GoDaddy               ┆ 4816 │
│ MER_000011  ┆ 38428 ┆ Atlassian             ┆ 7372 │
│ MER_000883  ┆ 38115

In [21]:
# Глобальное пересечение мерчантов (не только топ-20)
b_merch = set(business.select('merchant_id').unique().to_series().to_list())
c_merch = set(consumer.select('merchant_id').unique().to_series().to_list())
both = b_merch & c_merch
print(f'Уникальных мерчантов в business: {len(b_merch):,}')
print(f'Уникальных мерчантов в consumer: {len(c_merch):,}')
print(f'Пересечение: {len(both):,}  (доля от business = {len(both)/max(len(b_merch),1):.1%})')

Уникальных мерчантов в business: 481
Уникальных мерчантов в consumer: 2,060
Пересечение: 376  (доля от business = 78.2%)


## 5. Качество данных

In [22]:
# 5.1 Дубликаты транзакций по ключевым полям
dup_keys = ['transaction_timestamp', 'transaction_amount_kzt', 'card_number', 'merchant_id']
for name, df in [('business', business), ('consumer', consumer)]:
    n_dup = df.height - df.unique(subset=dup_keys).height
    print(f'{name}: {n_dup:,} полных дубликатов по {dup_keys}')

business: 0 полных дубликатов по ['transaction_timestamp', 'transaction_amount_kzt', 'card_number', 'merchant_id']
consumer: 0 полных дубликатов по ['transaction_timestamp', 'transaction_amount_kzt', 'card_number', 'merchant_id']


In [23]:
# 5.2 Карты, которые присутствуют и в business, и в consumer — критично для PU-split
b_cards_set = set(business.select('card_number').unique().to_series().to_list())
c_cards_set = set(consumer.select('card_number').unique().to_series().to_list())
card_overlap = b_cards_set & c_cards_set
print(f'business unique cards: {len(b_cards_set):,}')
print(f'consumer unique cards: {len(c_cards_set):,}')
print(f'Пересечение по card_number: {len(card_overlap):,}')
if card_overlap:
    print('!!! Утечка: одни и те же карты в обоих датасетах — фатально для PU-learning split !!!')

business unique cards: 25,000
consumer unique cards: 80,000
Пересечение по card_number: 0


In [24]:
# 5.3 Выбросы — что выше 99-го перцентиля
for name, df in [('business', business), ('consumer', consumer)]:
    p99 = df.select(pl.col('transaction_amount_kzt').quantile(0.99)).item()
    mx = df.select(pl.col('transaction_amount_kzt').max()).item()
    over = df.filter(pl.col('transaction_amount_kzt') > p99).height
    print(f'{name}: p99 = {p99:,.0f} ₸ | max = {mx:,.0f} ₸ | tx > p99 = {over:,}')

business: p99 = 1,090,844 ₸ | max = 40,799,297 ₸ | tx > p99 = 29,976
consumer: p99 = 699,869 ₸ | max = 31,971,032 ₸ | tx > p99 = 98,324


In [25]:
# 5.4 Нулевые или отрицательные суммы
for name, df in [('business', business), ('consumer', consumer)]:
    n_zero = df.filter(pl.col('transaction_amount_kzt') == 0).height
    n_neg = df.filter(pl.col('transaction_amount_kzt') < 0).height
    print(f'{name}: zero = {n_zero:,} | negative = {n_neg:,}')

business: zero = 0 | negative = 0
consumer: zero = 0 | negative = 0


In [27]:
# 5.5 Доля B2B и Consumer MCC по гипотезе FEATURES.md
# mcc в данных — String, а в config.py коды — int. Приводим к строкам.
def mcc_class_share(df: pl.DataFrame) -> dict:
    total = df.height
    b2b = [str(c) for c in B2B_MCC_CODES]
    cns = [str(c) for c in CONSUMER_MCC_CODES]
    mcc_str = pl.col('mcc').cast(pl.Utf8)
    return {
        'b2b_share': df.filter(mcc_str.is_in(b2b)).height / total,
        'consumer_share': df.filter(mcc_str.is_in(cns)).height / total,
        'mcc_dtype': str(df.schema['mcc']),
    }

print('business:', mcc_class_share(business))
print('consumer:', mcc_class_share(consumer))


business: {'b2b_share': 0.3413098442650487, 'consumer_share': 0.07252885898786127, 'mcc_dtype': 'String'}
consumer: {'b2b_share': 0.04228919906021742, 'consumer_share': 0.2056936358014, 'mcc_dtype': 'String'}


## Выводы EDA

*(Шаблон, ориентированный на ожидаемые наблюдения из docs/FEATURES.md. После первого запуска — сверить с реальными цифрами и при расхождении переписать.)*

### Главные различия business vs consumer

- **Объём:** business-карты делают заметно больше транзакций на карту и тратят больше денег (см. card-level boxplots в §3.2). Средний чек у business тяжелее в правом хвосте — крупные оптовые закупки.
- **Diversity:** у business выше число уникальных мерчантов и MCC на карту — типичная картина деловых закупок у многих поставщиков.
- **Каналы:** доля online у business выше (SaaS, реклама, эквайринговые сервисы); tokenized — наоборот ниже (Apple/Samsung Pay — потребительский паттерн).
- **Recurring:** у business выше — это SaaS-подписки и сервисные платежи.
- **Время:** business сильно сконцентрирован в рабочие часы 9–18 пн-пт; consumer — вечером и в выходные.
- **MCC-микс:** топ business насыщен B2B-кодами из `B2B_MCC_CODES` (7372/7311/5300/...), топ consumer — 5411/5812/5814 (магазины, рестораны).

### Подтверждённые гипотезы из docs/FEATURES.md

- ⭐ **Группа C (B2B MCC):** доля транзакций на B2B-MCC у business в разы выше — подтверждено через `mcc_class_share()`. Сильнейший одиночный сигнал, как и заявлено в приоритете №1.
- ⭐ **Группа D (временные паттерны):** business_hours_share значимо выше у business — гипотеза подтверждена графиком §3.4.
- **Группа B (diversity):** n_unique_merchants и n_unique_mcc выше у business — оправдывает HHI и top-K-share фичи.
- **Группа E (recurring):** recurring_share выше у business — оправдывает b2b_recurring_share.
- **Группа F (online):** online_share у business выше — оправдывает online_b2b_share как интеракцию.

### Неожиданные находки

- *(заполнить после запуска)* — например, аномальная сезонность в декабре/январе; нетривиальная доля consumer-MCC у business-карт (рестораны для оплаты деловых обедов?); большое пересечение мерчантов между классами — значит сами по себе merchant_id плохо разделяют, нужна агрегатная статистика.

### Красные флаги в данных

- **Пересечение card_number между business и consumer** — если > 0, нужно исключать такие карты из train-сплита (см. §5.2).
- **Дубликаты транзакций** — если есть, повлияют на агрегаты; решить — оставлять или дедуплицировать.
- **Нулевые/отрицательные суммы** — выбросить на этапе feature engineering.
- **Тяжёлый правый хвост сумм** — для линейных моделей нужно log1p; для LightGBM не критично.
- **Синтетика** — артефакты генератора могут давать слишком чистые сигналы. Митигация на этапе модели через SHAP.